In [1]:
import os
print(os.getcwd())
os.chdir('/home/user1/MXY/EHRScore') # Change to the project directory

/home/user1/MXY/EHRScore/data/mimic3


In [2]:
import pandas as pd

# read formatted mimic3 data
os.makedirs("data/mimic3/preprocessed", exist_ok=True)
data_path = "data/mimic3/format_mimic3_anonymized.csv"
mimic3_df = pd.read_csv(data_path)
mimic3_df

,PatientID,VisitID,RecordTime,RecordTimestamp,AdmissionTime,DischargeTime,Outcome,LOS,Readmission,Sex,...,Temperature,Oxygen saturation,Fraction inspired oxygen,Glucose,PH,Conditions_ICD9,Conditions_Long,Medications,Procedures_ICD9,Procedures_Long
0,10001,1,1,2159-09-13 11:58:00,2159-09-13 11:58:00,2159-09-18 14:07:00,0,5.09,1,1,...,NaN,NaN,NaN,NaN,NaN,4241;2851;4412;4019;2720;V1084,Aortic valve disorders;Acute posthemorrhagic a...,Simvastatin;0.9% Sodium Chloride;Sterile Water...,3722;3845;3521;8856;3893;3961,Left heart cardiac catheterization;Resection o...
1,10001,1,2,2159-09-13 23:58:00,2159-09-13 11:58:00,2159-09-18 14:07:00,0,5.09,1,1,...,NaN,NaN,NaN,124.0,7.60,NaN,NaN,NaN,NaN,NaN
2,10001,1,3,2159-09-14 11:58:00,2159-09-13 11:58:00,2159-09-18 14:07:00,0,5.09,1,1,...,36.80,98.0,NaN,137.0,7.49,NaN,NaN,NaN,NaN,NaN
3,10001,1,4,2159-09-14 23:58:00,2159-09-13 11:58:00,2159-09-18 14:07:00,0,5.09,1,1,...,37.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10001,1,5,2159-09-15 11:58:00,2159-09-13 11:58:00,2159-09-18 14:07:00,0,5.09,1,1,...,36.94,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1224447,56445,1,2,2143-12-16 08:14:00,2143-12-15 20:14:00,2143-12-18 15:52:00,0,2.82,0,0,...,37.61,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1224448,56445,1,3,2143-12-16 20:14:00,2143-12-15 20:14:00,2143-12-18 15:52:00,0,2.82,0,0,...,37.17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1224449,56445,1,4,2143-12-17 08:14:00,2143-12-15 20:14:00,2143-12-18 15:52:00,0,2.82,0,0,...,36.83,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1224450,56445,1,5,2143-12-17 20:14:00,2143-12-15 20:14:00,2143-12-18 15:52:00,0,2.82,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
import numpy as np

# ================= CONFIG =================
TASK                     = "outcome"   # "outcome" or "readmission"
RNG_SEED                 = 42
DESIRED_TOTAL            = 30_000 # outcome: mimic3:30_000 readmission: mimic3:16_000

# Outcome
DESIRED_POS_RATIO_OUTCOME = 0.13 

# Readmission
TARGET_RATE_READM   = 0.17  
TOLERANCE_READM     = 0.005  
# =========================================

if TASK == "outcome":
    filtered_df = (
        mimic3_df.groupby("PatientID").filter(
            lambda x: len(x) >= 10
        )
    )
elif TASK == "readmission":
    filtered_df = (
        mimic3_df.groupby("PatientID").filter(
            lambda x: len(x) >= 10
        )
    )
print(f"[INFO] {filtered_df['PatientID'].nunique():,} patients remaining after filtering")

rng = np.random.default_rng(RNG_SEED)

# Outcome TASK

if TASK.lower() == "outcome":
    label_col = "Outcome"

    desired_pos        = int(DESIRED_TOTAL * DESIRED_POS_RATIO_OUTCOME)   
    desired_neg        = DESIRED_TOTAL - desired_pos   

    patient_label_df = (
    filtered_df.groupby("PatientID")[label_col]
    .max()                   
    .reset_index()
)

    pos_patients = patient_label_df[patient_label_df[label_col] == 1]["PatientID"].to_numpy()
    neg_patients = patient_label_df[patient_label_df[label_col] == 0]["PatientID"].to_numpy()
    
    rng = np.random.default_rng(seed=42)     
    sampled_pos = rng.choice(pos_patients, size=min(desired_pos, len(pos_patients)), replace=False)
    sampled_neg = rng.choice(neg_patients, size=min(desired_neg, len(neg_patients)), replace=False)
    
    selected_patients = np.concatenate([sampled_pos, sampled_neg])
    print(f"Total number of patients: {len(selected_patients)},  Positive sample: {len(sampled_pos)}, Negative samples: {len(sampled_neg)}")
    
    # Patients after retention of sampling
    filtered_df = filtered_df[filtered_df["PatientID"].isin(selected_patients)]
    
    # Take the first 48 records for each patient.
    mimic3_filtered_df = filtered_df.groupby(['PatientID', 'VisitID']).head(48)

# Readmission TASK

elif TASK.lower() == "readmission":
    readmit_col = "Readmission"
    visit_labels = {}           
    patient_labels = {}        
    processed_parts = []

    for pid, grp in filtered_df.groupby("PatientID"):
        grp = grp.sort_values("VisitID")        
        visits = grp["VisitID"].unique()

        visit_flag = (
            grp.groupby("VisitID")[readmit_col]
               .max().reindex(visits, fill_value=0).to_numpy()
        )
    
        last_one_idx = np.where(visit_flag == 1)[0][-1] if visit_flag.any() else None
        keep_visits = visits if last_one_idx is None else visits[: last_one_idx + 1]

        has_positive = False
        for vid, flag in zip(visits, visit_flag):
            if vid in keep_visits:
                visit_labels[(pid, vid)] = int(flag)
                if flag == 1:
                    has_positive = True
        patient_labels[pid] = int(has_positive)

        processed_parts.append(grp[grp["VisitID"].isin(keep_visits)])

    processed_df = pd.concat(processed_parts, ignore_index=True)

    pos_pids = [pid for pid, l in patient_labels.items() if l == 1]
    neg_pids = [pid for pid, l in patient_labels.items() if l == 0]

    desired_pos = int(DESIRED_TOTAL * TARGET_RATE_READM)
    desired_neg = DESIRED_TOTAL - desired_pos

    sampled_pos = rng.choice(pos_pids, size=min(desired_pos, len(pos_pids)), replace=False)
    sampled_neg = rng.choice(neg_pids, size=min(desired_neg, len(neg_pids)), replace=False)

    if len(sampled_pos) < desired_pos:
        need = desired_pos - len(sampled_pos)
        extra_neg = rng.choice(
            [i for i in neg_pids if i not in sampled_neg],
            size=min(need, len(neg_pids) - len(sampled_neg)),
            replace=False,
        )
        sampled_neg = np.concatenate([sampled_neg, extra_neg])

    if len(sampled_neg) < desired_neg:
        need = desired_neg - len(sampled_neg)
        extra_pos = rng.choice(
            [i for i in pos_pids if i not in sampled_pos],
            size=min(need, len(pos_pids) - len(sampled_pos)),
            replace=False,
        )
        sampled_pos = np.concatenate([sampled_pos, extra_pos])

    selected_pids = np.concatenate([sampled_pos, sampled_neg])
    rng.shuffle(selected_pids)

    actual_rate = len(sampled_pos) / len(selected_pids)
    print(
        f"[Readmission] Selected {len(selected_pids):,} patients, "
        f"Readmission rate: {actual_rate:.3%} (Target: {TARGET_RATE_READM:.3%})"
    )

    mimic3_filtered_df = (
        processed_df[processed_df["PatientID"].isin(selected_pids)]
        .groupby(["PatientID", "VisitID"])
        .head(48)
        .reset_index(drop=True)
    )


[INFO] 32,155 patients remaining after filtering
Total number of patients: 28698,  Positive sample: 3900, Negative samples: 24798


In [4]:
# Perform LOCF(Last Observation Carried Forward) imputation
exclude_cols = [
    'Conditions_ICD9',
    'Conditions_Long',
    'Medications',
    'Procedures_ICD9',
    'Procedures_Long'
]

cols_to_impute = [col for col in mimic3_filtered_df.columns if col not in exclude_cols]

# locf
locf_imputed_df = mimic3_filtered_df.copy()
locf_imputed_df[cols_to_impute] = locf_imputed_df.groupby(['PatientID', 'VisitID'])[cols_to_impute].ffill()

# nocb
nocb_imputed_df = locf_imputed_df.copy()
nocb_imputed_df[cols_to_impute] = nocb_imputed_df.groupby(['PatientID', 'VisitID'])[cols_to_impute].bfill()

nocb_imputed_df

,PatientID,VisitID,RecordTime,RecordTimestamp,AdmissionTime,DischargeTime,Outcome,LOS,Readmission,Sex,...,Temperature,Oxygen saturation,Fraction inspired oxygen,Glucose,PH,Conditions_ICD9,Conditions_Long,Medications,Procedures_ICD9,Procedures_Long
0,10001,1,1,2159-09-13 11:58:00,2159-09-13 11:58:00,2159-09-18 14:07:00,0,5.09,1,1,...,36.80,98.0,NaN,124.0,7.60,4241;2851;4412;4019;2720;V1084,Aortic valve disorders;Acute posthemorrhagic a...,Simvastatin;0.9% Sodium Chloride;Sterile Water...,3722;3845;3521;8856;3893;3961,Left heart cardiac catheterization;Resection o...
1,10001,1,2,2159-09-13 23:58:00,2159-09-13 11:58:00,2159-09-18 14:07:00,0,5.09,1,1,...,36.80,98.0,NaN,124.0,7.60,NaN,NaN,NaN,NaN,NaN
2,10001,1,3,2159-09-14 11:58:00,2159-09-13 11:58:00,2159-09-18 14:07:00,0,5.09,1,1,...,36.80,98.0,NaN,137.0,7.49,NaN,NaN,NaN,NaN,NaN
3,10001,1,4,2159-09-14 23:58:00,2159-09-13 11:58:00,2159-09-18 14:07:00,0,5.09,1,1,...,37.50,98.0,NaN,137.0,7.49,NaN,NaN,NaN,NaN,NaN
4,10001,1,5,2159-09-15 11:58:00,2159-09-13 11:58:00,2159-09-18 14:07:00,0,5.09,1,1,...,36.94,98.0,NaN,137.0,7.49,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1224432,56443,1,12,2148-01-01 10:35:00,2147-12-26 22:35:00,2148-01-03 14:31:00,0,7.66,0,1,...,36.50,78.0,NaN,NaN,7.52,NaN,NaN,NaN,NaN,NaN
1224433,56443,1,13,2148-01-01 22:35:00,2147-12-26 22:35:00,2148-01-03 14:31:00,0,7.66,0,1,...,36.50,78.0,NaN,NaN,7.52,NaN,NaN,NaN,NaN,NaN
1224434,56443,1,14,2148-01-02 10:35:00,2147-12-26 22:35:00,2148-01-03 14:31:00,0,7.66,0,1,...,36.50,78.0,NaN,NaN,7.52,NaN,NaN,NaN,NaN,NaN
1224435,56443,1,15,2148-01-02 22:35:00,2147-12-26 22:35:00,2148-01-03 14:31:00,0,7.66,0,1,...,36.50,78.0,NaN,NaN,7.52,NaN,NaN,NaN,NaN,NaN


In [5]:
# Save the time-series data for subsequent statistical analysis
timeseries_df = nocb_imputed_df[cols_to_impute]
timeseries_df.to_csv("data/mimic3/preprocessed/timeseries_mimic3.csv", index=False)
timeseries_df

,PatientID,VisitID,RecordTime,RecordTimestamp,AdmissionTime,DischargeTime,Outcome,LOS,Readmission,Sex,...,Diastolic blood pressure,Systolic blood pressure,Mean blood pressure,Heart Rate,Respiratory rate,Temperature,Oxygen saturation,Fraction inspired oxygen,Glucose,PH
0,10001,1,1,2159-09-13 11:58:00,2159-09-13 11:58:00,2159-09-18 14:07:00,0,5.09,1,1,...,74.0,139.0,97.0,76.0,19.0,36.80,98.0,NaN,124.0,7.60
1,10001,1,2,2159-09-13 23:58:00,2159-09-13 11:58:00,2159-09-18 14:07:00,0,5.09,1,1,...,74.0,139.0,97.0,76.0,19.0,36.80,98.0,NaN,124.0,7.60
2,10001,1,3,2159-09-14 11:58:00,2159-09-13 11:58:00,2159-09-18 14:07:00,0,5.09,1,1,...,74.0,139.0,97.0,76.0,19.0,36.80,98.0,NaN,137.0,7.49
3,10001,1,4,2159-09-14 23:58:00,2159-09-13 11:58:00,2159-09-18 14:07:00,0,5.09,1,1,...,53.0,123.0,73.0,77.0,16.0,37.50,98.0,NaN,137.0,7.49
4,10001,1,5,2159-09-15 11:58:00,2159-09-13 11:58:00,2159-09-18 14:07:00,0,5.09,1,1,...,54.0,139.0,80.0,81.0,18.0,36.94,98.0,NaN,137.0,7.49
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1224432,56443,1,12,2148-01-01 10:35:00,2147-12-26 22:35:00,2148-01-03 14:31:00,0,7.66,0,1,...,NaN,NaN,NaN,91.0,33.0,36.50,78.0,NaN,NaN,7.52
1224433,56443,1,13,2148-01-01 22:35:00,2147-12-26 22:35:00,2148-01-03 14:31:00,0,7.66,0,1,...,NaN,NaN,NaN,91.0,33.0,36.50,78.0,NaN,NaN,7.52
1224434,56443,1,14,2148-01-02 10:35:00,2147-12-26 22:35:00,2148-01-03 14:31:00,0,7.66,0,1,...,NaN,NaN,NaN,91.0,33.0,36.50,78.0,NaN,NaN,7.52
1224435,56443,1,15,2148-01-02 22:35:00,2147-12-26 22:35:00,2148-01-03 14:31:00,0,7.66,0,1,...,NaN,NaN,NaN,91.0,33.0,36.50,78.0,NaN,NaN,7.52


In [6]:
# Extract the patient's conditions, medications, and procedures to CMPs.csv
clinical_cols = [
    'PatientID', 'VisitID',
    'AdmissionTime', 'DischargeTime', 
    'Conditions_ICD9', 'Conditions_Long',
    'Medications',
    'Procedures_ICD9', 'Procedures_Long'
]
cmps_df = nocb_imputed_df.groupby(['PatientID', 'VisitID']).first().reset_index()[clinical_cols]

cmps_df.to_csv("data/mimic3/preprocessed/CMPs_mimic3.csv", index=False)
cmps_df

,PatientID,VisitID,AdmissionTime,DischargeTime,Conditions_ICD9,Conditions_Long,Medications,Procedures_ICD9,Procedures_Long
0,10001,1,2159-09-13 11:58:00,2159-09-18 14:07:00,4241;2851;4412;4019;2720;V1084,Aortic valve disorders;Acute posthemorrhagic a...,Simvastatin;0.9% Sodium Chloride;Sterile Water...,3722;3845;3521;8856;3893;3961,Left heart cardiac catheterization;Resection o...
1,10001,2,2159-09-20 22:15:00,2159-10-04 15:24:00,99859;5192;5849;2761;42090;E8781;4019;2720;041...,Other postoperative infection;Mediastinitis;Ac...,CefazoLIN;0.9% Sodium Chloride;SW;CefazoLIN;Ca...,3403,Reopening of recent thoracotomy site
2,10005,1,2190-09-19 19:20:00,2190-10-01 15:52:00,5789;5849;42833;51881;41071;2851;5854;4160;403...,"Hemorrhage of gastrointestinal tract, unspecif...",Morphine Sulfate;Furosemide;Metoprolol Succina...,9390;4513;4523,Non-invasive mechanical ventilation;Other endo...
3,10007,1,2147-02-07 02:26:00,2147-02-17 18:40:00,431;5990;3314;2761;496;3485;25060;3572;41401;2...,Intracerebral hemorrhage;Urinary tract infecti...,Soln.;Mannitol 20%;Sodium Chloride 0.9% Flush...,None,None
4,10008,1,2107-09-18 18:38:00,2107-09-25 14:40:00,42832;5990;43820;4280;78701;25000;43813;43889;...,Chronic diastolic heart failure;Urinary tract ...,Bisacodyl;Amlodipine;Metoprolol Tartrate;Metop...,None,None
...,...,...,...,...,...,...,...,...,...
38596,56440,4,2195-10-09 19:53:00,2195-10-13 12:15:00,49121;51884;42833;2864;5849;32723;4168;4019;4280,Obstructive chronic bronchitis with (acute) ex...,Albuterol 0.083% Neb Soln;Lisinopril;Torsemide...,None,None
38597,56441,1,2156-10-20 08:00:00,2156-11-11 15:16:00,1541;51851;56722;5180;5601;99749;2851;2762;293...,Malignant neoplasm of rectum;Acute respiratory...,Chlorhexidine Gluconate 0.12% Oral Rinse;Fenta...,4594;5421;4601;4869;5252;415;1742;5491;9910;9729,Large-to-large intestinal anastomosis;Laparosc...
38598,56442,1,2163-06-10 19:48:00,2163-06-14 13:15:00,41051;41401;4019;2724,Acute myocardial infarction of other lateral w...,Sodium Chloride 0.9% Flush;Captopril;Metoprol...,8856;0066;3606;0041;0046;3722,Coronary arteriography using two catheters;Per...
38599,56442,2,2164-03-09 15:14:00,2164-03-16 13:10:00,41401;4111;412;4019;2724;27801;56400;4293;V4582,Coronary atherosclerosis of native coronary ar...,SW;Atropine Sulfate;Atorvastatin;Aspirin;Sodiu...,8856;3611;3722;3615;8853,Coronary arteriography using two catheters;(Ao...
